# Reading Catalyst Query Plans

## Spark 3.5+ logical and physical plan semantics

How to read every line, symbol, exchange, aggregate phase, join strategy, partitioning requirement, and adaptive-plan marker.

# Learning goals

You will learn to:

- distinguish parsed, analyzed, optimized, physical, and adaptive plans;
- read plan trees from the leaves upward;
- decode `Project`, `Filter`, `Exchange`, partitioning, sorting, joins, and aggregates;
- interpret `*`, `+-`, `:-`, `#42`, `ENSURE_REQUIREMENTS`, `isFinalPlan`, and statistics;
- connect physical operators to jobs, stages, tasks, shuffles, and generated code; and
- diagnose unnecessary movement, skew, weak estimates, and missed optimizations.

# A query does not have only one plan

![Catalyst plan representations](assets/plan_pipeline.svg)

Each representation answers a different question. The logical plans describe **what** result is required. The physical plan describes **how** Spark intends to obtain it. AQE can revise the remaining physical work after runtime statistics arrive.

## The five plan views

| View | Main semantics | Typical clues |
|---|---|---|
| Parsed logical | Syntax converted to unresolved nodes | `'table`, `'column`, unresolved aliases |
| Analyzed logical | Relations, columns, functions, and types resolved | `column#id`, casts, qualified attributes |
| Optimized logical | Equivalent relational expression after rules | pushed filters, pruned columns, folded constants |
| Physical | Chosen executable operators | `Scan`, `Exchange`, `HashAggregate`, join algorithms |
| Adaptive executed | Runtime-refined physical plan | `AdaptiveSparkPlan`, `ShuffleQueryStage`, `AQEShuffleRead` |

Use `df.explain("extended")` for all major plan forms and `df.explain("formatted")` for readable physical-node details.

# All DataFrame `explain` modes

Spark 3.5 supports five named modes through `df.explain(mode=...)`:

| Mode | Expected output | Best use |
|---|---|---|
| `simple` | Only the physical plan | Quick check of scans, joins, aggregates, sorts, and exchanges |
| `extended` | Parsed, analyzed, optimized logical plans, followed by the physical plan | Understand how Catalyst resolved and rewrote a query |
| `codegen` | Whole-stage codegen subtrees and generated Java code | Inspect operator fusion and generated execution code |
| `cost` | Optimized logical plan with statistics plus the physical plan | Inspect row/size estimates used for planning |
| `formatted` | Physical-plan outline followed by numbered node-detail sections | Read a large physical plan and operator arguments clearly |

These modes print diagnostic text and return `None`; they do not execute the DataFrame action or return a plan object. With AQE enabled, a plan printed before an action can show `isFinalPlan=false`.

## Mode-by-mode output expectations

### `simple`

```text
== Physical Plan ==
AdaptiveSparkPlan ...
+- HashAggregate ...
   +- Exchange ...
```

Expect one compact physical operator tree. It omits the earlier logical-plan sections. This is the default for `df.explain()` and `df.explain(False)`.

## `extended` output expectation

```text
== Parsed Logical Plan ==
... unresolved syntax-level tree ...
== Analyzed Logical Plan ==
... resolved attributes and data types ...
== Optimized Logical Plan ==
... rule-rewritten relational plan ...
== Physical Plan ==
... chosen executable operators ...
```

Use it to prove where a filter moved, whether projections were pruned, where casts appeared, and how logical nodes became physical operators. `df.explain(True)` is the legacy boolean shorthand for `extended`.

## `codegen` output expectation

```text
Found N WholeStageCodegen subtrees.
== Subtree 1 / N ... ==
*(1) Project ...
+- *(1) Filter ...
Generated code:
... generated Java source ...
```

Expect zero or more fused subtrees and their generated JVM code. A result of `Found 0 WholeStageCodegen subtrees` is valid—for example, when the current plan/operators cannot participate. The `*(n)` labels identify codegen pipelines, not scheduler stages. The generated code is verbose and is mainly useful for advanced diagnosis.

## `cost` output expectation

```text
== Optimized Logical Plan ==
Aggregate ..., Statistics(sizeInBytes=..., rowCount=...)
+- Filter ..., Statistics(sizeInBytes=..., rowCount=...)
== Physical Plan ==
... chosen executable operators ...
```

Expect optimizer statistics beside logical nodes, then the physical plan. `sizeInBytes` is generally present; `rowCount` and column statistics appear only when Spark can derive or obtain them. These are estimates—not observed runtime metrics. Missing or stale catalog statistics can make the values weak.

## `formatted` output expectation

```text
== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- Range (1)

(3) Exchange
Input [2]: [...]
Arguments: hashpartitioning(...), ENSURE_REQUIREMENTS, ...
```

Expect a compact outline plus a details block for each numbered operator. The numbers connect outline nodes to detail sections; they are not jobs, scheduler stages, task IDs, or codegen IDs. This is usually the easiest mode for line-by-line physical-plan teaching.

# API forms and SQL equivalents

```python
df.explain()                  # simple
df.explain(False)             # simple; legacy boolean form
df.explain(True)              # extended; legacy boolean form
df.explain(mode="simple")
df.explain(mode="extended")
df.explain(mode="codegen")
df.explain(mode="cost")
df.explain(mode="formatted")
```

SQL provides the corresponding forms: `EXPLAIN`, `EXPLAIN EXTENDED`, `EXPLAIN CODEGEN`, `EXPLAIN COST`, and `EXPLAIN FORMATTED` before a query. Use one named mode at a time. An unsupported mode raises an error instead of silently falling back.

In [ ]:
# Run after the Spark setup and an example DataFrame named `aggregated` exist.
# Uncomment one line at a time to keep notebook output readable.
# aggregated.explain(mode="simple")
# aggregated.explain(mode="extended")
# aggregated.explain(mode="codegen")
# aggregated.explain(mode="cost")
# aggregated.explain(mode="formatted")

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("CatalystQueryPlans")
         .master("local[4]")  # remove on a managed cluster
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.version

# How to read a plan tree

Read from the **bottom leaves upward** because data flows from scans toward the root result. Indentation expresses parent/child ownership.

```text
*(2) HashAggregate(keys=[country#12], functions=[sum(amount#13)])  ← root/output
+- Exchange hashpartitioning(country#12, 8), ENSURE_REQUIREMENTS  ← child
   +- *(1) HashAggregate(keys=[country#12], functions=[partial_sum(amount#13)])
      +- *(1) Filter (amount#13 > 0.0)
         +- *(1) Scan ExistingRDD[country#12,amount#13]            ← leaf/input
```

Narrate it upward: scan rows → filter them → locally pre-aggregate → repartition by country → merge aggregates.

## What do `+-`, `:-`, and indentation mean?

These are tree-drawing characters, not executable operators.

- `+-` introduces the last/only child of a node.
- `:-` introduces a non-final child when a node has multiple children.
- `|` continues a vertical branch through deeper levels.
- spaces preserve nesting/alignment.

```text
SortMergeJoin [customer_id#1], [customer_id#8], Inner
:- Sort ...                 ← left child
:  +- Exchange ...
:     +- Scan left ...
+- Sort ...                 ← right and final child
   +- Exchange ...
      +- Scan right ...
```

# Other symbols and identifiers

| Marker | Meaning |
|---|---|
| `'col` | Unresolved attribute in an early logical plan |
| `col#42` | Resolved Catalyst attribute with internal expression ID 42 |
| `AS total#91` | Alias creates a new output attribute/expression ID |
| `*` before node | Operator participates in whole-stage code generation |
| `*(3)` | Codegen pipeline/stage ID 3, **not** a Spark scheduler stage ID |
| `[]` | Expression/key/output lists or empty arguments, depending on position |
| `...L` | Long integer literal in plan rendering |
| `cast(x as type)` | Explicit or analyzer-inserted type conversion |
| `isnotnull(x)` | Null guard, sometimes optimizer-generated |

Expression IDs distinguish attributes even when display names match—for example, the left and right `id` in a self-join. Do not treat them as source column positions.

# Catalyst internals

Catalyst represents plans and expressions as immutable trees. Transformations use tree-pattern matching and rule batches.

```text
TreeNode
├── LogicalPlan
│   ├── Project / Filter / Aggregate / Join
│   └── unresolved and resolved expressions
└── SparkPlan
    ├── ProjectExec / FilterExec / HashAggregateExec
    ├── SortMergeJoinExec / BroadcastHashJoinExec
    └── ExchangeExec / ScanExec
```

The **Analyzer** resolves names and types using catalog/function information. The **Optimizer** repeatedly applies semantics-preserving logical rules. **SparkPlanner** strategies produce physical candidates. **EnsureRequirements** inserts exchanges/sorts where child output does not satisfy an operator's required distribution or ordering.

# `Project`: relational projection

`Project` selects, reorders, derives, casts, or aliases columns. It is not a UI display operation.

```text
Project [customer_id#1, (amount#2 * 1.18) AS gross#9]
+- Filter (amount#2 > 0.0)
   +- Relation [customer_id#1,amount#2,status#3] parquet
```

Catalyst can collapse adjacent projections and prune unused source columns. In the physical plan, `Project` may become `ProjectExec` and commonly fuses with filter/scan operators through whole-stage codegen. A `Project` alone is narrow and requires no shuffle.

# `Filter` and pushed filters

`Filter condition` evaluates SQL three-valued logic: only `TRUE` survives; `FALSE` and `NULL` are removed.

A data-source scan may display:

- `PartitionFilters`: predicates used to prune directory/table partitions;
- `PushedFilters`: predicates offered to the source for lower-level filtering;
- `DataFilters`: filters associated with scanned data; and
- `ReadSchema`: columns actually requested after pruning.

Seeing a pushed filter does not always mean Spark can remove its own filter—the source API may report whether filtering is exact. Verify rows and scan metrics rather than inferring behavior from the label alone.

In [ ]:
orders = (spark.range(0, 1_000_000, 1, 8)
          .select(
              F.col("id").alias("order_id"),
              (F.col("id") % 1000).alias("customer_id"),
              (F.col("id") % 10).alias("country_id"),
              (F.col("id") * 0.01).alias("amount")))

project_filter = (orders
                  .filter(F.col("amount") > 100)
                  .select("country_id",
                          (F.col("amount") * 1.18).alias("gross")))
project_filter.explain(mode="extended")

# `Exchange`: data distribution changes

`Exchange` is a physical operator that changes partition placement/distribution so its parent operator receives data in the required shape.

Common forms:

- `Exchange hashpartitioning(keys, N)`: shuffle equal keys to the same one of N partitions;
- `Exchange rangepartitioning(ordering, N)`: sample boundaries, then shuffle ordered key ranges;
- `Exchange SinglePartition`: move all records to one partition;
- `BroadcastExchange HashedRelationBroadcastMode(...)`: collect/build a small relation and broadcast it;
- `ReusedExchange`: reuse an equivalent previously planned/materialized exchange.

A shuffle exchange creates a data-transfer boundary. Operators below it write shuffle blocks; operators above it read them. A broadcast exchange has different mechanics but is still a distribution exchange.

## `ENSURE_REQUIREMENTS` and other exchange origins

Modern formatted plans can include an exchange origin:

- `ENSURE_REQUIREMENTS`: Spark inserted it because a parent operator required a distribution/order the child did not provide;
- `REPARTITION_BY_NUM`: caused by an explicit `repartition(number, ...)`;
- `REPARTITION_BY_COL`: caused by repartitioning expressions without an explicit partition count;
- `REBALANCE_PARTITIONS_BY_NONE/COL`: associated with rebalance hints;
- `COALESCE_PARTITIONS`: associated with a coalesce repartitioning request when represented as an exchange.

Names can vary by plan-rendering path and Spark maintenance release. The semantic question is: **was this exchange required by an operator, or explicitly requested by the query?**

# Shuffle is a mechanism, not a logical keyword

A shuffle redistributes records between executors/partitions. Map-side tasks write one logical block per downstream partition; reduce-side tasks fetch their block from every map output.

![Shuffle block redistribution](assets/shuffle_blocks.svg)

It entails serialization, partition computation, possible sort/spill, local disk files, network fetch, and merge/deserialization. `Exchange` is the plan node; **shuffle** is the distributed execution it requests.

# Hash partitioning

`hashpartitioning(k1, k2, N)` means Spark computes a partition from the SQL hash of the key expressions and maps it into `N` buckets. Its key guarantee is **co-location of equal keys**, not sorting.

```text
partitionId = nonNegativeModulo(hash(key expressions), N)

same key → same partition
different key ⇏ different partition
partition contents are not globally or locally sorted by this guarantee
```

Hash partitioning supports grouped aggregation and hash-based joins. Skewed keys can overload one partition. Both sides of a partitioned join must have compatible clustering—not merely the same displayed partition count.

# Range partitioning and sort semantics

`rangepartitioning(key ASC NULLS FIRST, N)` assigns ordered key ranges to partitions, using sampled data to estimate boundaries. It enables global ordering when paired with local sorting.

- **`Sort [...], true`**: global sort semantics; planning normally requires range partitioning plus sorting inside partitions.
- **`Sort [...], false`**: local/per-partition sort only; no global ordering guarantee.
- `ASC/DESC` controls direction.
- `NULLS FIRST/LAST` defines null placement.

```text
Exchange rangepartitioning(k ASC NULLS FIRST, 8)
+- input
Sort [k ASC NULLS FIRST], true
+- Exchange ...
```

Range partition boundaries are not necessarily equal-width; sampling aims for balanced record counts, subject to skew and estimation.

In [ ]:
hash_demo = orders.repartition(8, "customer_id")
range_demo = orders.orderBy(F.col("amount").desc())
local_sort_demo = orders.sortWithinPartitions("customer_id")

print("=== HASH REPARTITION ===")
hash_demo.explain(mode="formatted")
print("\n=== GLOBAL ORDER BY ===")
range_demo.explain(mode="formatted")
print("\n=== LOCAL SORT ===")
local_sort_demo.explain(mode="formatted")

# `partial_*` and final aggregation

A distributive/algebraic aggregate is commonly executed in two physical phases to reduce shuffle volume.

```text
Input rows
  → HashAggregate functions=[partial_sum(amount)]    local combine per map partition
  → Exchange hashpartitioning(groupKey, 8)           move compact buffers by key
  → HashAggregate functions=[sum(amount)]            merge buffers / final output
```

`partial_sum`, `partial_count`, and similar labels mean **update an intermediate aggregation buffer**, not an approximate or incomplete answer returned to the user. The final operator merges buffers and evaluates the final value. Average uses multiple buffer fields such as sum and count.

## Aggregate modes and operator choices

Internally, aggregate expressions have modes such as:

- **Partial**: raw input → intermediate buffer;
- **PartialMerge**: merge intermediate buffers;
- **Final**: buffers → final value;
- **Complete**: raw input → final value without a separate merge phase.

Physical operators include `HashAggregate`, `ObjectHashAggregate`, and `SortAggregate`. Choice depends on aggregate buffer/types, configuration, and planning support. A hash aggregate may fall back to sort-based processing under memory pressure; do not infer zero sorting/spilling solely from its name.

In [ ]:
aggregated = (orders
              .groupBy("country_id")
              .agg(
                  F.sum("amount").alias("revenue"),
                  F.avg("amount").alias("avg_order"),
                  F.count("*").alias("orders")))
aggregated.explain(mode="extended")

# Read upward: Range → Project → partial aggregate → Exchange → final aggregate.

# Join operator vocabulary

| Operator | Data preparation | Typical fit |
|---|---|---|
| `BroadcastHashJoin` | Broadcast build side; stream other side | One side is sufficiently small |
| `SortMergeJoin` | Hash-shuffle both sides by keys, then sort each side | Large equi-joins |
| `ShuffledHashJoin` | Hash-shuffle both sides; build local hash table | One shuffled side is much smaller per partition |
| `BroadcastNestedLoopJoin` | Broadcast one side; nested comparisons | Non-equi/cross patterns when a side is small |
| `CartesianProduct` | Cartesian pairing | Explicit/derived cross join |

`BuildLeft`/`BuildRight` identifies the side used to construct a hash relation. `Inner`, `LeftOuter`, `LeftSemi`, `LeftAnti`, etc. express join semantics. Physical strategy can change under AQE.

## Anatomy of a sort-merge join

```text
SortMergeJoin [customer_id#1], [customer_id#20], Inner
:- Sort [customer_id#1 ASC NULLS FIRST], false
:  +- Exchange hashpartitioning(customer_id#1, 8), ENSURE_REQUIREMENTS
:     +- Filter isnotnull(customer_id#1)
:        +- Scan orders
+- Sort [customer_id#20 ASC NULLS FIRST], false
   +- Exchange hashpartitioning(customer_id#20, 8), ENSURE_REQUIREMENTS
      +- Filter isnotnull(customer_id#20)
         +- Scan customers
```

Both sides become compatibly clustered and locally ordered. Null filters may be inferred because ordinary equality does not match null keys. The two exchange branches may execute concurrently before the join stage.

In [ ]:
customers = spark.range(0, 1000, 1, 4).select(
    F.col("id").alias("customer_id"),
    (F.col("id") % 5).alias("segment"))

old_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    sort_merge = orders.join(customers, "customer_id")
    sort_merge.explain(mode="formatted")
finally:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", old_threshold)

# Scan vocabulary

You may see `FileScan parquet`, `BatchScan`, `Scan ExistingRDD`, `Range`, `InMemoryTableScan`, or version/source-specific scan names. Important fields include:

- `Output` / `ReadSchema`: columns produced;
- `Location`: file index and paths;
- `PartitionFilters`: partition pruning;
- `PushedFilters`: predicates passed into the source;
- `DataFilters`: data-level filter expressions;
- `Batched`: vectorized/batched reader indication;
- `InMemoryRelation` and `InMemoryTableScan`: cached columnar data; and
- `RuntimeFilters` / dynamic pruning expressions: values determined using another query branch.

V1 and V2 data sources render differently. Focus on semantic fields, not memorizing one exact string.

# Window and limit operators

A window typically requires compatible clustering and ordering:

```text
Window [row_number() ...], [country_id#3], [amount#4 DESC]
+- Sort [country_id#3 ASC, amount#4 DESC], false
   +- Exchange hashpartitioning(country_id#3, 8), ENSURE_REQUIREMENTS
```

`LocalLimit n` limits each partition; `GlobalLimit n` enforces the overall limit, often after `Exchange SinglePartition`. For ordered top-N, Spark may use `TakeOrderedAndProject` rather than globally sorting every row. `CollectLimit` is a physical limit that returns a bounded result to the driver.

A warning about moving all window data to one partition means no partition key was supplied; this is correct semantics but potentially dangerous scalability.

# Reuse, subqueries, and dynamic pruning

- `Subquery` / `SubqueryBroadcast`: separately evaluated scalar or broadcast subplan.
- `ReusedSubquery`: equivalent subquery result is shared.
- `ReusedExchange`: duplicate exchange output is shared.
- `dynamicpruningexpression(...)`: use values from one relation to prune partitions of another at runtime.
- `SubqueryAdaptiveBroadcast`: adaptive broadcast-backed runtime filtering.

Reuse avoids repeated physical work within compatible execution. It also explains why the visible plan can contain references to already materialized query stages and why some scheduler stages appear skipped.

# Adaptive plan vocabulary

With AQE enabled, look for:

- `AdaptiveSparkPlan isFinalPlan=false`: current plan before execution has finalized;
- `isFinalPlan=true`: executed adaptive plan is available after the action;
- `ShuffleQueryStage` / `BroadcastQueryStage`: materializable AQE subplans;
- `AQEShuffleRead coalesced`: small shuffle partitions combined for fewer downstream tasks;
- `AQEShuffleRead skewed`: skew handling changed partition reads;
- `LocalShuffleReader`: read shuffle data locally when distribution constraints permit;
- `Final Plan` and `Initial Plan`: post- and pre-adaptation comparison.

An AQE query stage is not identical to a `DAGScheduler` stage. One is a SQL adaptive-planning unit; the other is an RDD scheduling unit separated by shuffle dependencies.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

adaptive = orders.groupBy("country_id").agg(F.sum("amount").alias("revenue"))
print("=== BEFORE ACTION ===")
adaptive.explain(mode="formatted")
adaptive.collect()
print("\n=== AFTER ACTION ===")
adaptive.explain(mode="formatted")

# Whole-stage code generation

Compatible physical operators can be fused into one generated Java method/pipeline, avoiding virtual calls and intermediate row objects.

```text
*(1) Project ...
+- *(1) Filter ...
   +- *(1) Scan ...

same *(1) → same codegen pipeline
Exchange → pipeline boundary
```

The star means codegen participation; the parenthesized number identifies a codegen pipeline. It is not a job or scheduler-stage number. Use `df.explain("codegen")` to inspect generated-code regions. Python UDF evaluation, exchanges, and some complex operators can break a pipeline.

In [ ]:
codegen_demo = (orders
                .filter(F.col("amount") > 10)
                .select("customer_id", (F.col("amount") * 2).alias("double_amount")))
codegen_demo.explain(mode="codegen")

# Compare the codegen subtree IDs with *(n) in the simple physical plan.

# Statistics and cost-based jargon

`EXPLAIN COST` / `df.explain("cost")` can show estimates such as:

- `sizeInBytes`: estimated output bytes;
- `rowCount`: estimated rows when available;
- attribute statistics: distinct count, null count, min/max, average/max length;
- logical-plan estimates used to compare some physical strategies.

Statistics may originate from catalog/table analysis, file metadata, data-source estimates, or defaults. Missing/stale estimates can cause a poor join order or missed/unsafe broadcast choice. AQE adds runtime map-output statistics but does not repair every upstream estimate.

`Cost` here means optimizer estimates, not cloud billing and not a guaranteed runtime prediction.

In [ ]:
aggregated.explain(mode="cost")

# For catalog tables, ANALYZE TABLE can populate table/column statistics.
# Syntax and support depend on the catalog and table provider in use.

# Plans, jobs, stages, and codegen IDs are different

```text
Catalyst logical operator tree
          ↓ selected implementation
SparkPlan physical operator tree
          ↓ execution produces RDD dependencies
DAGScheduler stages split at shuffle dependencies
          ↓ one task per required stage partition
Task attempts on executors
```

A physical plan can predict where shuffles occur, but it does not directly print scheduler stage IDs. `*(2)` is a codegen ID. `plan_id=42` identifies a plan node in some renderings. `ShuffleQueryStage 1` is an AQE ID. The Spark UI's `Stage 7` is a scheduler stage ID. Never equate these numbers.

# A repeatable plan-reading method

1. **Start at leaves:** Which relations, files, ranges, or caches are read?
2. **Check reduction early:** Were columns pruned and filters pushed below expensive work?
3. **Trace rows upward:** State each operator's input and output semantics.
4. **Circle exchanges:** What distribution is created, why, and with how many partitions?
5. **Inspect ordering:** Is it global or local, explicit or required by join/window?
6. **Decode aggregation phases:** raw rows, partial buffers, shuffle, final merge.
7. **Inspect joins:** type, keys, build side, broadcast/shuffle, null filters.
8. **Check estimates/AQE:** Are statistics plausible? What changed at runtime?
9. **Run and correlate:** Compare the executed plan with SQL and stage metrics in the UI.

A plan tells you semantics and intent; metrics tell you scale and actual cost.

# Diagnostic patterns

| Plan/metric clue | Likely question |
|---|---|
| Repeated `Exchange` on same keys | Can partitioning/reuse eliminate redundant movement? |
| `Exchange SinglePartition` over large input | Did a global operation collapse parallelism? |
| Sort-merge join with a truly tiny side | Are statistics missing or broadcast disabled? |
| Broadcast of a large relation | Is the estimate stale or threshold/hint unsafe? |
| One shuffle task much slower/larger | Is there key skew? |
| Large spill metrics | Is aggregation/sort/join exceeding execution memory? |
| Python UDF boundary | Can built-in SQL expressions restore optimization/codegen? |
| Full scan despite selective condition | Is pushdown/pruning supported and expressed correctly? |
| Many tiny post-shuffle tasks | Is AQE coalescing enabled/effective? |

Avoid optimizing only the visual plan. Confirm input rows/bytes, shuffle read/write, spill, task distribution, and runtime.

# Compact jargon glossary

- **Predicate:** Boolean expression used for filtering or joining.
- **Attribute:** A resolved column expression, identified internally by expression ID.
- **Projection:** Selection/derivation of output expressions.
- **Distribution:** How rows are placed across partitions (`Clustered`, `Ordered`, `Single`, etc.).
- **Ordering:** Sort guarantees within or across partitions.
- **Exchange:** Physical redistribution/broadcast operator.
- **Shuffle:** Distributed transfer implementing repartitioning.
- **Build side:** Join input turned into an in-memory hash relation.
- **Codegen pipeline:** Fused operators compiled into generated JVM code.
- **Canonicalization/reuse:** Normalize equivalent plans/expressions so work can be shared.
- **Query stage:** AQE materialization boundary around exchange work.
- **Spill:** Execution data written to disk when it cannot remain in memory.

# Final mental model

```text
WHAT?     logical plan: relation algebra + expressions
              ↓ analyze names/types
BETTER?   optimized logical plan: equivalent, cheaper expression
              ↓ choose implementations
HOW?      physical plan: scans, joins, aggregates, sorts, exchanges
              ↓ execute and measure
ACTUAL?   adaptive executed plan + Spark UI runtime metrics
```

Read upward from scans. Treat `Exchange` as a distribution boundary, `Sort` as an ordering guarantee, partial/final aggregates as buffer phases, and symbols as tree/codegen annotations. Then verify every performance conclusion using executed-plan and stage metrics.

# Practice lab

For each query, predict and then inspect parsed, analyzed, optimized, initial physical, and final adaptive plans:

1. `filter → select`: find pruning, predicate simplification, and codegen fusion.
2. `groupBy → sum/avg`: label partial buffers, exchange, and final merge.
3. `orderBy` versus `sortWithinPartitions`: compare range exchange and local ordering.
4. Large-to-small join with broadcast enabled/disabled: compare both physical trees.
5. Window by key and order: identify required clustered distribution and sort order.
6. Cache and reuse the result: find `InMemoryTableScan` and skipped work.
7. Create a heavily skewed key: compare initial and final AQE plans plus task metrics.

Explain every line in plain English before discussing performance. Semantics come first.